# SEA-TauBench Paper Plots

Reusable notebook for generating paper-ready plots from the filtered SEA-TauBench result CSV.

- Input CSV: `/Users/saksornr/Code/tau3_result_plot/result_consolidate.csv`
- Output figures: `/Users/saksornr/Code/tau3_result_plot/figures`


## Configuration

Change paths, filter settings, metric selections, labels, and export formats here when reusing the notebook with updated data.


In [51]:
from pathlib import Path
import os

PROJECT_DIR = Path("/Users/saksornr/Code/tau3_result_plot")
CSV_PATH = PROJECT_DIR / "result_consolidate.csv"
FIG_DIR = PROJECT_DIR / "figures"

CACHE_DIR = PROJECT_DIR / ".cache"
MPL_CONFIG_DIR = CACHE_DIR / "matplotlib"
for directory in (FIG_DIR, CACHE_DIR, MPL_CONFIG_DIR, CACHE_DIR / "fontconfig"):
    directory.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("MPLCONFIGDIR", str(MPL_CONFIG_DIR))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR))

EXPORT_FORMATS = ("pdf", "png", "svg")
EXPORT_DPI = 400

FILTER_SETTING = {
    "senario": [
        "1: english only",
        "2: multilingual tools",
        "3: crosslingual",
        "4: translated",
    ],
    "domain": ["airline", "retail", "telecom"],
    "normalized language/senario": [
        "english",
        "chinese",
        "indonesian",
        "thai",
        "vietnamese",
        "filipino",
        "tool_mix_2",
        "tool_mix_3",
        "tool_mix_4",
        "tool_mix_5",
    ],
    "normalized agent model": [
        "gpt-5-mini",
        "qwen3-235b-a22b-inst",
        "qwen3.6-35b-a3b",
        "kimi-k2.5",
    ],
}

PRIMARY_METRICS = ["pass@1", "pass@2", "pass@3"]
PAPER_METRICS = ["pass@1", "pass@3"]
FIGURE5_METRICS = ["pass@1", "rho@3"]
FIGURE6_METRICS = ["pass@1", "rho@3"]
ACTION_METRICS = ["read_actions", "write_actions", "db_match", "language_correctness"]

METRIC_RENAMES = {
    "pass^1": "pass@1",
    "pass^2": "pass@2",
    "pass^3": "pass@3",
    "rho^3": "rho@3",
    "Read Actions": "read_actions",
    "Write Actions": "write_actions",
    "DB Match": "db_match",
    "language_correctness": "language_correctness",
}

SCENARIO_LABELS = {
    1: "EN Baseline",
    2: "L2 Tools",
    3: "L2 Interaction",
    4: "L2 Domain",
}
SCENARIO_ORDER = [1, 3, 2, 4]
NON_BASELINE_SCENARIO_ORDER = [3, 2, 4]

LANGUAGE_ORDER = [
    "english",
    "thai",
    "vietnamese",
    "filipino",
    "indonesian",
    "chinese",
]
TOOL_MIX_ORDER = ["tool_mix_2", "tool_mix_3", "tool_mix_4", "tool_mix_5"]
LANGUAGE_LABELS = {
    "english": "EN",
    "chinese": "ZH",
    "indonesian": "ID",
    "thai": "TH",
    "vietnamese": "VI",
    "filipino": "FL",
    "tool_mix_2": "Mix 2",
    "tool_mix_3": "Mix 3",
    "tool_mix_4": "Mix 4",
    "tool_mix_5": "Mix 5",
}
LANGUAGE_DISPLAY_NAMES = {
    "english": "English",
    "chinese": "Chinese",
    "indonesian": "Indonesian",
    "thai": "Thai",
    "vietnamese": "Vietnamese",
    "filipino": "Filipino",
    "tool_mix_2": "Tool Mix 2",
    "tool_mix_3": "Tool Mix 3",
    "tool_mix_4": "Tool Mix 4",
    "tool_mix_5": "Tool Mix 5",
}

MODEL_ORDER = FILTER_SETTING["normalized agent model"]
PLOT_MODEL_KEYS = ["gpt-5-mini", "qwen3-235b-a22b-inst"]
FIG6_ALL_MODEL_SCENARIO_IDS = {3, 4}
MODEL_LABELS = {
    "gpt-5-mini": "GPT-5 Mini",
    "kimi-k2.5": "Kimi K2.5",
    "qwen3-235b-a22b-inst": "Qwen3-235B-A22B-INST",
    "qwen3.6-35b-a3b": "Qwen3.6-35B-A3B",
}

print(f"CSV_PATH: {CSV_PATH}")
print(f"FIG_DIR: {FIG_DIR}")
print("FILTER_SETTING:")
for key, values in FILTER_SETTING.items():
    print(f"  {key}: {values}")
print(f"Default plot model subset: {PLOT_MODEL_KEYS}")
print(f"Figure 6 all-model scenario IDs: {sorted(FIG6_ALL_MODEL_SCENARIO_IDS)}")


CSV_PATH: /Users/saksornr/Code/tau3_result_plot/result_consolidate.csv
FIG_DIR: /Users/saksornr/Code/tau3_result_plot/figures
FILTER_SETTING:
  senario: ['1: english only', '2: multilingual tools', '3: crosslingual', '4: translated']
  domain: ['airline', 'retail', 'telecom']
  normalized language/senario: ['english', 'chinese', 'indonesian', 'thai', 'vietnamese', 'filipino', 'tool_mix_2', 'tool_mix_3', 'tool_mix_4', 'tool_mix_5']
  normalized agent model: ['gpt-5-mini', 'qwen3-235b-a22b-inst', 'qwen3.6-35b-a3b', 'kimi-k2.5']
Default plot model subset: ['gpt-5-mini', 'qwen3-235b-a22b-inst']
Figure 6 all-model scenario IDs: [3, 4]


## Imports and Plot Style

The style uses colorblind-safe colors, vector-friendly font settings, compact paper proportions, and matplotlib-only plotting so the notebook runs in this local environment.


In [52]:
import math

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

OKABE_ITO = {
    "orange": "#E69F00",
    "sky_blue": "#56B4E9",
    "bluish_green": "#009E73",
    "yellow": "#F0E442",
    "blue": "#0072B2",
    "vermillion": "#D55E00",
    "reddish_purple": "#CC79A7",
    "black": "#000000",
}

LANGUAGE_PALETTE = {
    "EN": OKABE_ITO["orange"],
    "ZH": OKABE_ITO["sky_blue"],
    "ID": OKABE_ITO["blue"],
    "TH": OKABE_ITO["bluish_green"],
    "VI": OKABE_ITO["reddish_purple"],
    "FL": OKABE_ITO["vermillion"],
}

METRIC_PALETTE = {
    "pass@1": OKABE_ITO["orange"],
    "pass@2": OKABE_ITO["sky_blue"],
    "pass@3": OKABE_ITO["reddish_purple"],
    "rho@3": OKABE_ITO["reddish_purple"],
}

MODEL_PALETTE = dict(zip(MODEL_ORDER, [
    OKABE_ITO["orange"],
    OKABE_ITO["sky_blue"],
    OKABE_ITO["bluish_green"],
    OKABE_ITO["reddish_purple"],
]))

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.dpi": 140,
        "savefig.dpi": EXPORT_DPI,
        "savefig.bbox": "tight",
        "savefig.facecolor": "white",
        "font.family": "DejaVu Sans",
        "font.size": 8,
        "axes.titlesize": 9,
        "axes.labelsize": 8,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "legend.fontsize": 7,
        "axes.linewidth": 0.7,
        "grid.linewidth": 0.45,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    }
)


def despine(ax):
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)


def save_publication_figure(fig, name, formats=EXPORT_FORMATS):
    saved_paths = []
    for ext in formats:
        out_path = FIG_DIR / f"{name}.{ext}"
        fig.savefig(out_path, dpi=EXPORT_DPI, bbox_inches="tight", facecolor="white")
        saved_paths.append(out_path)
    print("Saved:")
    for path in saved_paths:
        print(f"  {path}")
    return saved_paths


## Load, Filter, Normalize, and Deduplicate

Data rules:

- Keep rows that match `FILTER_SETTING` exactly.
- Use `normalized agent model` and `normalized language/senario` as the only model and language/scenario sources.
- Keep rows that have at least one primary pass metric.
- Deduplicate by scenario/domain/normalized language-scenario/normalized model, preferring `Finalized=True`, then the latest CSV row.


In [53]:
def normalize_key_series(series):
    return series.astype("string").str.strip().str.lower()


def coerce_finalized(value):
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def parse_scenario_id(series):
    return pd.to_numeric(
        series.astype("string").str.extract(r"^\s*(\d+)")[0],
        errors="coerce",
    ).astype("Int64")


def load_and_prepare(csv_path=CSV_PATH):
    raw = pd.read_csv(csv_path)
    raw.columns = [column.strip() for column in raw.columns]
    df = raw.reset_index(names="source_row").copy()

    for original, renamed in METRIC_RENAMES.items():
        if original in df.columns:
            df[renamed] = pd.to_numeric(df[original], errors="coerce")
        else:
            df[renamed] = np.nan

    df["scenario_raw"] = df["Senario"].astype("string").str.strip()
    df["scenario_id"] = parse_scenario_id(df["scenario_raw"])
    df["scenario_label"] = df["scenario_id"].map(SCENARIO_LABELS).fillna(df["scenario_raw"])

    df["domain_key"] = normalize_key_series(df["domain"])
    df["domain_label"] = df["domain_key"].str.title()
    df["language_key"] = normalize_key_series(df["normalized language/senario"])
    df["language_label"] = df["language_key"].map(LANGUAGE_LABELS).fillna(df["language_key"])
    df["language_display"] = df["language_key"].map(LANGUAGE_DISPLAY_NAMES).fillna(df["language_key"])
    df["language_group"] = np.where(df["language_key"].str.startswith("tool_mix_", na=False), "tool_mix", "language")
    df["tool_mix_n"] = pd.to_numeric(
        df["language_key"].str.extract(r"tool_mix_(\d+)")[0],
        errors="coerce",
    )

    df["model_key"] = normalize_key_series(df["normalized agent model"])
    df["model_label"] = df["model_key"].map(MODEL_LABELS).fillna(df["model_key"])
    df["finalized_bool"] = df["Finalized"].map(coerce_finalized)

    has_primary_metric = df[PRIMARY_METRICS].notna().any(axis=1)
    filter_mask = (
        df["scenario_raw"].isin(FILTER_SETTING["senario"])
        & df["domain_key"].isin(FILTER_SETTING["domain"])
        & df["language_key"].isin(FILTER_SETTING["normalized language/senario"])
        & df["model_key"].isin(FILTER_SETTING["normalized agent model"])
    )
    with_metrics = df.loc[filter_mask & has_primary_metric].copy()

    dedupe_keys = ["scenario_raw", "domain_key", "language_key", "model_key"]
    duplicate_summary = (
        with_metrics.groupby(dedupe_keys, dropna=False)
        .size()
        .reset_index(name="row_count")
        .query("row_count > 1")
        .sort_values("row_count", ascending=False)
    )
    clean = (
        with_metrics.sort_values(["finalized_bool", "source_row"])
        .drop_duplicates(dedupe_keys, keep="last")
        .sort_values("source_row")
        .reset_index(drop=True)
    )

    audit = {
        "raw_rows": len(raw),
        "rows_matching_filter": int(filter_mask.sum()),
        "filtered_rows_with_primary_metrics_before_dedupe": len(with_metrics),
        "clean_rows_after_dedupe": len(clean),
        "duplicate_groups_resolved": len(duplicate_summary),
        "excluded_rows_by_filter_or_missing_metrics": len(df) - len(with_metrics),
    }
    return raw, df, clean, duplicate_summary, audit


raw_df, normalized_df, clean_df, duplicate_summary, audit = load_and_prepare()
plot_df = clean_df.loc[clean_df["model_key"].isin(PLOT_MODEL_KEYS)].copy()

print("Audit")
for key, value in audit.items():
    print(f"  {key}: {value}")

print("\nClean scenarios:")
print(clean_df.groupby(["scenario_id", "scenario_label"]).size().to_string())

print("\nClean domains:")
print(clean_df.groupby("domain_key").size().to_string())

print("\nClean models available for Figure 6:")
print(clean_df.groupby("model_key").size().reindex(MODEL_ORDER).dropna().astype(int).to_string())

print("\nDefault plot model subset:")
print(plot_df.groupby("model_key").size().reindex(PLOT_MODEL_KEYS).dropna().astype(int).to_string())

print("\nDuplicate groups resolved:")
if duplicate_summary.empty:
    print("  None")
else:
    print(duplicate_summary.to_string(index=False))

preview_cols = [
    "scenario_label",
    "domain_label",
    "language_label",
    "model_label",
    "pass@1",
    "pass@2",
    "pass@3",
    "rho@3",
    "finalized_bool",
]
try:
    display(clean_df[preview_cols].head(12))
except NameError:
    print(clean_df[preview_cols].head(12).to_string(index=False))


Audit
  raw_rows: 187
  rows_matching_filter: 174
  filtered_rows_with_primary_metrics_before_dedupe: 173
  clean_rows_after_dedupe: 170
  duplicate_groups_resolved: 3
  excluded_rows_by_filter_or_missing_metrics: 14

Clean scenarios:
scenario_id  scenario_label
1            EN Baseline       12
2            L2 Tools          54
3            L2 Interaction    60
4            L2 Domain         44

Clean domains:
domain_key
airline    62
retail     62
telecom    46

Clean models available for Figure 6:
model_key
gpt-5-mini              59
qwen3-235b-a22b-inst    55
qwen3.6-35b-a3b         28
kimi-k2.5               28

Default plot model subset:
model_key
gpt-5-mini              59
qwen3-235b-a22b-inst    55

Duplicate groups resolved:
   scenario_raw domain_key language_key  model_key  row_count
1: english only    airline      english gpt-5-mini          2
1: english only     retail      english gpt-5-mini          2
1: english only    telecom      english gpt-5-mini          2


,scenario_label,domain_label,language_label,model_label,pass@1,pass@2,pass@3,rho@3,finalized_bool
0,EN Baseline,Airline,EN,GPT-5 Mini,0.693,0.593,0.540,0.779,True
1,EN Baseline,Airline,EN,Qwen3-235B-A22B-INST,0.500,0.387,0.340,0.680,True
2,EN Baseline,Airline,EN,Kimi K2.5,0.707,0.600,0.540,0.764,True
3,EN Baseline,Airline,EN,Qwen3.6-35B-A3B,0.773,0.663,0.580,0.750,True
4,EN Baseline,Retail,EN,GPT-5 Mini,0.655,0.497,0.412,0.629,True
5,EN Baseline,Retail,EN,Qwen3-235B-A22B-INST,0.579,0.415,0.342,0.591,True
6,EN Baseline,Retail,EN,Qwen3.6-35B-A3B,0.655,0.515,0.439,0.670,True
7,EN Baseline,Retail,EN,Kimi K2.5,0.561,0.442,0.395,0.704,True
8,EN Baseline,Telecom,EN,GPT-5 Mini,0.713,0.588,0.509,0.714,True
9,EN Baseline,Telecom,EN,Qwen3-235B-A22B-INST,0.357,0.205,0.132,0.370,True


## Figure 5: Degradation by Language and Setting

Each panel shows one non-English language. The first point repeats the English-only baseline so the degradation trend has a stable reference.


In [54]:
def available_language_order(frame):
    available = set(frame.loc[frame["language_group"].eq("language"), "language_key"].dropna())
    ordered = [language for language in LANGUAGE_ORDER if language in available]
    extras = sorted(available.difference(ordered))
    return ordered + extras


TARGET_LANGUAGES = [
    language
    for language in available_language_order(plot_df)
    if language != "english"
]
AVAILABLE_SCENARIO_IDS = set(plot_df["scenario_id"].dropna().astype(int))
FIGURE5_SCENARIOS = [scenario for scenario in NON_BASELINE_SCENARIO_ORDER if scenario in AVAILABLE_SCENARIO_IDS]
CONDITION_ORDER = ["EN Baseline"] + [SCENARIO_LABELS[scenario] for scenario in FIGURE5_SCENARIOS]
CONDITION_SHORT_LABELS = {
    "EN Baseline": "EN Baseline",
    "L2 Tools": "L2 Tools",
    "L2 Interaction": "L2 Interaction",
    "L2 Domain": "L2 Domain",
}


def build_figure5_frame(frame):
    baseline = frame.loc[
        frame["scenario_id"].eq(1)
        & frame["language_key"].eq("english")
        & frame["language_group"].eq("language")
    ].copy()
    baseline["condition"] = "EN Baseline"

    expanded_baseline = []
    for language in TARGET_LANGUAGES:
        copy = baseline.copy()
        copy["panel_language"] = language
        copy["panel_label"] = LANGUAGE_LABELS.get(language, language)
        expanded_baseline.append(copy)

    language_rows = frame.loc[
        frame["scenario_id"].isin(FIGURE5_SCENARIOS)
        & frame["language_key"].isin(TARGET_LANGUAGES)
        & frame["language_group"].eq("language")
    ].copy()
    language_rows["condition"] = language_rows["scenario_id"].map(SCENARIO_LABELS)
    language_rows["panel_language"] = language_rows["language_key"]
    language_rows["panel_label"] = language_rows["language_label"]

    parts = expanded_baseline + [language_rows]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


fig5_df = build_figure5_frame(plot_df)
FIGURE5_METRIC_LABELS = {"pass@1": "pass@1", "rho@3": r"$\rho^3$"}
FIGURE5_METRIC_PALETTE = {"pass@1": METRIC_PALETTE["pass@1"], "rho@3": METRIC_PALETTE["rho@3"]}

n_panels = max(1, len(TARGET_LANGUAGES))
n_cols = min(3, n_panels)
n_rows = math.ceil(n_panels / n_cols)
fig = plt.figure(figsize=(2.55 * n_cols, 2.15 * n_rows))
grid = fig.add_gridspec(n_rows, n_cols * 2)
axes_flat = []
for panel_idx in range(len(TARGET_LANGUAGES)):
    row_idx = panel_idx // n_cols
    col_idx = panel_idx % n_cols
    panels_in_row = min(n_cols, len(TARGET_LANGUAGES) - row_idx * n_cols)
    row_offset = n_cols - panels_in_row
    start_col = col_idx * 2 + row_offset
    share_axis = axes_flat[0] if axes_flat else None
    axes_flat.append(fig.add_subplot(grid[row_idx, start_col : start_col + 2], sharey=share_axis))
x_positions = np.arange(len(CONDITION_ORDER))
score_ticks = np.linspace(0, 1, 6)
score_tick_labels = [f"{tick:g}" for tick in score_ticks]

for ax, language in zip(axes_flat, TARGET_LANGUAGES):
    subset = fig5_df.loc[fig5_df["panel_language"].eq(language)]
    for metric in FIGURE5_METRICS:
        stats = (
            subset.groupby("condition")[metric]
            .agg(["mean", "std", "count"])
            .reindex(CONDITION_ORDER)
        )
        mean = stats["mean"]
        sem = stats["std"].fillna(0) / np.sqrt(stats["count"].clip(lower=1))
        y_values = mean.to_numpy(dtype=float).copy()
        y_errors = sem.to_numpy(dtype=float).copy()
        y_errors[np.isnan(y_values)] = np.nan
        ax.errorbar(
            x_positions,
            y_values,
            yerr=y_errors,
            marker="o",
            markersize=4,
            linewidth=1.6,
            capsize=2.5,
            color=FIGURE5_METRIC_PALETTE[metric],
            label=FIGURE5_METRIC_LABELS[metric],
        )

    ax.set_title(f"{LANGUAGE_LABELS.get(language, language)} ({LANGUAGE_DISPLAY_NAMES.get(language, language)})")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([CONDITION_SHORT_LABELS[label] for label in CONDITION_ORDER], rotation=20, ha="right")
    ax.set_ylim(0, 1.02)
    ax.set_yticks(score_ticks)
    ax.set_yticklabels(score_tick_labels)
    ax.tick_params(axis="y", labelleft=True)
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.35)
    despine(ax)

handles, labels = axes_flat[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=len(labels),
        frameon=False,
    )
fig.tight_layout(rect=[0, 0.10, 1, 1])
save_publication_figure(fig, "figure5_degradation_by_language")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/figure5_degradation_by_language.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/figure5_degradation_by_language.png
  /Users/saksornr/Code/tau3_result_plot/figures/figure5_degradation_by_language.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/1416278399.py:122: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 7: Degradation by Language with Model Dots

Each panel follows Figure 5 and overlays model-level averages as dots. Dots use the same color as the corresponding metric line.


In [55]:
FIGURE7_METRICS = FIGURE5_METRICS
FIGURE7_METRIC_LABELS = FIGURE5_METRIC_LABELS
FIGURE7_METRIC_PALETTE = FIGURE5_METRIC_PALETTE
FIGURE7_ALL_MODEL_SCENARIO_IDS = {3, 4}


def figure7_allowed_models(condition):
    if condition == "EN Baseline":
        return MODEL_ORDER
    scenario_id = next(
        (key for key, value in SCENARIO_LABELS.items() if value == condition),
        None,
    )
    if scenario_id in FIGURE7_ALL_MODEL_SCENARIO_IDS:
        return MODEL_ORDER
    return PLOT_MODEL_KEYS


def build_figure7_frame(frame):
    figure7_frame = build_figure5_frame(frame).copy()
    allowed_parts = []
    for condition in CONDITION_ORDER:
        allowed_models = set(figure7_allowed_models(condition))
        allowed_parts.append(
            figure7_frame.loc[
                figure7_frame["condition"].eq(condition)
                & figure7_frame["model_key"].isin(allowed_models)
            ]
        )
    return pd.concat(allowed_parts, ignore_index=True) if allowed_parts else pd.DataFrame()


fig7_df = build_figure7_frame(clean_df)

n_panels = max(1, len(TARGET_LANGUAGES))
n_cols = min(3, n_panels)
n_rows = math.ceil(n_panels / n_cols)
fig = plt.figure(figsize=(2.55 * n_cols, 2.15 * n_rows))
grid = fig.add_gridspec(n_rows, n_cols * 2)
axes_flat = []
for panel_idx in range(len(TARGET_LANGUAGES)):
    row_idx = panel_idx // n_cols
    col_idx = panel_idx % n_cols
    panels_in_row = min(n_cols, len(TARGET_LANGUAGES) - row_idx * n_cols)
    row_offset = n_cols - panels_in_row
    start_col = col_idx * 2 + row_offset
    share_axis = axes_flat[0] if axes_flat else None
    axes_flat.append(fig.add_subplot(grid[row_idx, start_col : start_col + 2], sharey=share_axis))

x_positions = np.arange(len(CONDITION_ORDER))
score_ticks = np.linspace(0, 1, 6)
score_tick_labels = [f"{tick:g}" for tick in score_ticks]
metric_offsets = {
    metric: offset
    for metric, offset in zip(FIGURE7_METRICS, np.linspace(-0.045, 0.045, len(FIGURE7_METRICS)))
}
model_offsets = {
    model: offset
    for model, offset in zip(MODEL_ORDER, np.linspace(-0.03, 0.03, len(MODEL_ORDER)))
}

for ax, language in zip(axes_flat, TARGET_LANGUAGES):
    subset = fig7_df.loc[fig7_df["panel_language"].eq(language)]
    for metric in FIGURE7_METRICS:
        stats = (
            subset.groupby("condition")[metric]
            .agg(["mean", "std", "count"])
            .reindex(CONDITION_ORDER)
        )
        mean = stats["mean"]
        sem = stats["std"].fillna(0) / np.sqrt(stats["count"].clip(lower=1))
        y_values = mean.to_numpy(dtype=float).copy()
        y_errors = sem.to_numpy(dtype=float).copy()
        y_errors[np.isnan(y_values)] = np.nan

        dot_summary = (
            subset.groupby(["condition", "model_key"])[metric]
            .mean()
            .reset_index()
        )
        for condition_idx, condition in enumerate(CONDITION_ORDER):
            allowed_models = figure7_allowed_models(condition)
            condition_dots = (
                dot_summary.loc[dot_summary["condition"].eq(condition)]
                .set_index("model_key")
                .reindex(allowed_models)
                .dropna(subset=[metric])
                .reset_index()
            )
            if condition_dots.empty:
                continue
            dot_x = np.array(
                [
                    x_positions[condition_idx]
                    + metric_offsets[metric]
                    + model_offsets.get(model, 0)
                    for model in condition_dots["model_key"]
                ],
                dtype=float,
            )
            ax.scatter(
                dot_x,
                condition_dots[metric].to_numpy(dtype=float),
                s=18,
                color=FIGURE7_METRIC_PALETTE[metric],
                edgecolor="white",
                linewidth=0.35,
                alpha=0.72,
                zorder=4,
            )
        ax.errorbar(
            x_positions,
            y_values,
            yerr=y_errors,
            marker="o",
            markersize=4,
            linewidth=1.6,
            capsize=2.5,
            color=FIGURE7_METRIC_PALETTE[metric],
            label=FIGURE7_METRIC_LABELS[metric],
            zorder=3,
        )

    ax.set_title(f"{LANGUAGE_LABELS.get(language, language)} ({LANGUAGE_DISPLAY_NAMES.get(language, language)})")
    ax.set_xticks(x_positions)
    ax.set_xticklabels([CONDITION_SHORT_LABELS[label] for label in CONDITION_ORDER], rotation=20, ha="right")
    ax.set_ylim(0, 1.02)
    ax.set_yticks(score_ticks)
    ax.set_yticklabels(score_tick_labels)
    ax.tick_params(axis="y", labelleft=True)
    ax.set_ylabel("Score")
    ax.grid(axis="y", alpha=0.35)
    despine(ax)

handles, labels = axes_flat[0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.015),
        ncol=len(labels),
        frameon=False,
    )
fig.tight_layout(rect=[0, 0.10, 1, 1])
save_publication_figure(fig, "figure7_degradation_by_language_with_model_dots")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/figure7_degradation_by_language_with_model_dots.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/figure7_degradation_by_language_with_model_dots.png
  /Users/saksornr/Code/tau3_result_plot/figures/figure7_degradation_by_language_with_model_dots.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/762837481.py:147: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Figure 6: Model and Language Breakdown

Rows are metrics and columns are available non-baseline settings. Missing combinations are skipped without failing.


In [69]:
def ordered_model_labels(frame):
    available = set(frame["model_key"].dropna())
    return [model for model in MODEL_ORDER if model in available]


MODEL_KEY_ORDER = ordered_model_labels(clean_df)
MODEL_LABEL_ORDER = [MODEL_LABELS.get(model, model) for model in MODEL_KEY_ORDER]
scenario_metric_availability = (
    clean_df.loc[clean_df["language_group"].eq("language")]
    .groupby("scenario_id")[PAPER_METRICS]
    .agg(lambda values: values.notna().any())
)
NON_BASELINE_SCENARIOS = [
    scenario
    for scenario in NON_BASELINE_SCENARIO_ORDER
    if scenario in scenario_metric_availability.index
    and bool(scenario_metric_availability.loc[scenario].any())
]
LANGUAGE_HUE_ORDER = [
    LANGUAGE_LABELS[language]
    for language in LANGUAGE_ORDER
    if language in set(clean_df["language_key"])
]
BASELINE_MODEL_KEYS = set(
    clean_df.loc[
        clean_df["scenario_id"].eq(1)
        & clean_df["language_key"].eq("english")
        & clean_df["language_group"].eq("language")
        & clean_df[FIGURE6_METRICS].notna().any(axis=1),
        "model_key",
    ]
)
FIG6_MODEL_ORDER_BY_SCENARIO = {}
FIG6_SCENARIO_SLOT_COUNTS = []
for scenario in NON_BASELINE_SCENARIOS:
    scenario_model_keys = set(
        clean_df.loc[
            clean_df["scenario_id"].eq(scenario)
            & clean_df["language_group"].eq("language")
            & clean_df[FIGURE6_METRICS].notna().any(axis=1),
            "model_key",
        ]
    )
    allowed_models = MODEL_KEY_ORDER if scenario in FIG6_ALL_MODEL_SCENARIO_IDS else [
        model for model in MODEL_KEY_ORDER if model in PLOT_MODEL_KEYS
    ]
    scenario_models = [
        model
        for model in allowed_models
        if model in scenario_model_keys or model in BASELINE_MODEL_KEYS
    ]
    FIG6_MODEL_ORDER_BY_SCENARIO[scenario] = scenario_models
    FIG6_SCENARIO_SLOT_COUNTS.append(max(1, len(scenario_models)))

fig, axes = plt.subplots(
    len(FIGURE6_METRICS),
    len(NON_BASELINE_SCENARIOS),
    figsize=(0.8 * sum(FIG6_SCENARIO_SLOT_COUNTS), 2.6875 * len(FIGURE6_METRICS)),
    sharey=True,
    squeeze=False,
    gridspec_kw={"width_ratios": FIG6_SCENARIO_SLOT_COUNTS},
)
fixed_bar_width = 0.12
hue_offsets = (np.arange(len(LANGUAGE_HUE_ORDER)) - (len(LANGUAGE_HUE_ORDER) - 1) / 2) * fixed_bar_width

FIGURE6_METRIC_LABELS = {"pass@1": "pass@1", "rho@3": r"$\rho^3$"}

for row_idx, metric in enumerate(FIGURE6_METRICS):
    for col_idx, scenario_id in enumerate(NON_BASELINE_SCENARIOS):
        ax = axes[row_idx, col_idx]
        scenario_subset = clean_df.loc[
            clean_df["scenario_id"].eq(scenario_id)
            & clean_df["language_group"].eq("language")
            & clean_df[metric].notna()
        ].copy()

        order = FIG6_MODEL_ORDER_BY_SCENARIO[scenario_id]
        english_baseline_subset = clean_df.loc[
            clean_df["scenario_id"].eq(1)
            & clean_df["language_key"].eq("english")
            & clean_df["model_key"].isin(order)
            & clean_df[metric].notna()
        ].copy()
        subset = pd.concat([english_baseline_subset, scenario_subset], ignore_index=True)
        if subset.empty:
            ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue

        x_positions = np.arange(len(order))
        summary = (
            subset.groupby(["model_key", "language_label"])[metric]
            .agg(["mean"])
            .reset_index()
        )

        for hue_idx, language_label in enumerate(LANGUAGE_HUE_ORDER):
            hue_summary = (
                summary.loc[summary["language_label"].eq(language_label)]
                .set_index("model_key")
                .reindex(order)
            )
            y_values = hue_summary["mean"].to_numpy(dtype=float).copy()
            valid = ~np.isnan(y_values)
            if not valid.any():
                continue
            bar_x = x_positions + hue_offsets[hue_idx]
            ax.bar(
                bar_x[valid],
                y_values[valid],
                width=fixed_bar_width * 0.9,
                color=LANGUAGE_PALETTE[language_label],
                edgecolor="white",
                linewidth=0.3,
            )

        ax.set_title(SCENARIO_LABELS.get(scenario_id, str(scenario_id)))
        ax.set_xlabel("")
        ax.set_ylabel(FIGURE6_METRIC_LABELS.get(metric, metric) if col_idx == 0 else "")
        ax.set_ylim(0, 1.02)
        ax.set_xticks(x_positions)
        ax.set_xticklabels([MODEL_LABELS.get(model, model) for model in order])
        ax.set_xlim(-0.55, max(0.55, len(order) - 0.45))
        ax.tick_params(axis="x", rotation=32)
        for label in ax.get_xticklabels():
            label.set_horizontalalignment("right")
        ax.grid(axis="y", alpha=0.35)
        despine(ax)

legend_handles = [
    Patch(facecolor=LANGUAGE_PALETTE[label], edgecolor="none")
    for label in LANGUAGE_HUE_ORDER
]
fig.legend(
    legend_handles,
    LANGUAGE_HUE_ORDER,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.015),
    ncol=min(6, len(LANGUAGE_HUE_ORDER)),
    frameon=False,
)
fig.tight_layout(rect=[0, 0.07, 1, 1])
save_publication_figure(fig, "figure6_model_language_breakdown")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/figure6_model_language_breakdown.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/figure6_model_language_breakdown.png
  /Users/saksornr/Code/tau3_result_plot/figures/figure6_model_language_breakdown.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/2962166740.py:144: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis Figure: Language-Scenario Heatmaps

Mean performance by language and scenario for the two headline metrics.


In [57]:
normal_language_df = plot_df.loc[plot_df["language_group"].eq("language")].copy()
language_row_order = [
    LANGUAGE_LABELS.get(language, language)
    for language in LANGUAGE_ORDER
    if language in set(normal_language_df["language_key"])
]
scenario_col_order = [
    SCENARIO_LABELS[scenario]
    for scenario in SCENARIO_ORDER
    if scenario in set(normal_language_df["scenario_id"].dropna().astype(int))
]

fig, axes = plt.subplots(1, len(PAPER_METRICS), figsize=(3.6 * len(PAPER_METRICS), 3.2), squeeze=False)
for ax, metric in zip(axes.ravel(), PAPER_METRICS):
    heatmap = (
        normal_language_df.groupby(["language_label", "scenario_label"])[metric]
        .mean()
        .unstack()
        .reindex(index=language_row_order, columns=scenario_col_order)
    )
    data = heatmap.to_numpy(dtype=float)
    image = ax.imshow(data, cmap="YlGnBu", vmin=0, vmax=1, aspect="auto")
    for row_idx in range(data.shape[0]):
        for col_idx in range(data.shape[1]):
            value = data[row_idx, col_idx]
            if not np.isnan(value):
                ax.text(col_idx, row_idx, f"{value:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(metric)
    ax.set_xticks(np.arange(len(scenario_col_order)))
    ax.set_xticklabels(scenario_col_order, rotation=25, ha="right")
    ax.set_yticks(np.arange(len(language_row_order)))
    ax.set_yticklabels(language_row_order)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label=metric)
fig.tight_layout()
save_publication_figure(fig, "analysis_heatmap_language_scenario")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.png
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/1782010851.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis Figure: Gap to English Baseline

Negative values indicate the average score falls below the English-only baseline.


In [58]:
english_baseline = normal_language_df.loc[
    normal_language_df["scenario_id"].eq(1)
    & normal_language_df["language_key"].eq("english")
][PAPER_METRICS].mean()

gap_source = (
    normal_language_df.loc[
        ~normal_language_df["language_key"].eq("english")
        & normal_language_df["scenario_id"].isin(NON_BASELINE_SCENARIO_ORDER)
    ]
    .groupby(["language_label", "scenario_label"])[PAPER_METRICS]
    .mean()
    .reset_index()
)
gap_long = gap_source.melt(
    id_vars=["language_label", "scenario_label"],
    value_vars=PAPER_METRICS,
    var_name="metric",
    value_name="score",
)
gap_long["gap_to_en_baseline"] = gap_long.apply(
    lambda row: row["score"] - english_baseline[row["metric"]],
    axis=1,
)
gap_limit = gap_long["gap_to_en_baseline"].abs().max()
gap_limit = 0.05 if pd.isna(gap_limit) else max(0.05, float(gap_limit) * 1.18)

fig, axes = plt.subplots(1, len(PAPER_METRICS), figsize=(3.7 * len(PAPER_METRICS), 2.9), sharey=True)
if len(PAPER_METRICS) == 1:
    axes = [axes]
scenario_hue_order = [label for label in scenario_col_order if label != "EN Baseline"]
bar_width = 0.8 / max(1, len(scenario_hue_order))
x_labels = [label for label in language_row_order if label != "EN"]
x_positions = np.arange(len(x_labels))
scenario_colors = dict(zip(scenario_hue_order, [OKABE_ITO["sky_blue"], OKABE_ITO["bluish_green"], OKABE_ITO["reddish_purple"]]))
for ax, metric in zip(axes, PAPER_METRICS):
    subset = gap_long.loc[gap_long["metric"].eq(metric)].dropna(subset=["gap_to_en_baseline"])
    for scenario_idx, scenario_label in enumerate(scenario_hue_order):
        values = (
            subset.loc[subset["scenario_label"].eq(scenario_label)]
            .set_index("language_label")
            .reindex(x_labels)["gap_to_en_baseline"]
            .to_numpy(dtype=float)
        )
        valid = ~np.isnan(values)
        offsets = x_positions - 0.4 + bar_width / 2 + scenario_idx * bar_width
        ax.bar(
            offsets[valid],
            values[valid],
            width=bar_width * 0.9,
            color=scenario_colors.get(scenario_label, OKABE_ITO["blue"]),
            label=scenario_label,
        )
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_ylim(-gap_limit, gap_limit)
    ax.set_title(metric)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(x_labels)
    ax.set_xlabel("")
    ax.set_ylabel("Gap to EN baseline" if ax is axes[0] else "")
    ax.grid(axis="y", alpha=0.35)
    despine(ax)

handles, labels = axes[-1].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=min(4, len(labels)),
        frameon=False,
    )
fig.tight_layout(rect=[0, 0.08, 1, 1])
save_publication_figure(fig, "analysis_gap_to_english")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.png
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/1368481801.py:76: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis Figure: Model Ranking

Average performance across all available cleaned rows.


In [59]:
ranking_long = plot_df.melt(
    id_vars=["model_key", "model_label"],
    value_vars=PAPER_METRICS,
    var_name="metric",
    value_name="score",
).dropna(subset=["score"])

ranking_order = (
    ranking_long.loc[ranking_long["metric"].eq("pass@1")]
    .groupby("model_key")["score"]
    .mean()
    .sort_values(ascending=True)
    .index.tolist()
)
y_positions = np.arange(len(ranking_order))
bar_height = 0.36

fig, ax = plt.subplots(figsize=(5.2, max(2.6, 0.42 * len(ranking_order))))
for metric_idx, metric in enumerate(PAPER_METRICS):
    stats = (
        ranking_long.loc[ranking_long["metric"].eq(metric)]
        .groupby("model_key")["score"]
        .agg(["mean", "std", "count"])
        .reindex(ranking_order)
    )
    sem = stats["std"].fillna(0) / np.sqrt(stats["count"].clip(lower=1))
    offsets = y_positions + (metric_idx - (len(PAPER_METRICS) - 1) / 2) * bar_height
    ax.barh(
        offsets,
        stats["mean"],
        xerr=sem,
        height=bar_height * 0.9,
        color=METRIC_PALETTE[metric],
        label=metric,
        error_kw={"linewidth": 0.8, "capsize": 2},
    )
ax.set_xlim(0, 1.02)
ax.set_yticks(y_positions)
ax.set_yticklabels([MODEL_LABELS.get(model, model) for model in ranking_order])
ax.set_xlabel("Mean score")
ax.set_ylabel("")
ax.legend(title="", frameon=False, loc="center left", bbox_to_anchor=(1.02, 0.5), ncol=1)
ax.grid(axis="x", alpha=0.35)
despine(ax)
fig.tight_layout()
save_publication_figure(fig, "analysis_model_ranking")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.png
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/1268874004.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis Figure: Domain Robustness

Mean performance by task domain, aggregated over available scenarios, languages, and models.


In [60]:
domain_long = normal_language_df.melt(
    id_vars=["domain_key", "domain_label"],
    value_vars=PAPER_METRICS,
    var_name="metric",
    value_name="score",
).dropna(subset=["domain_key", "score"])
domain_order = (
    domain_long.loc[domain_long["metric"].eq("pass@1")]
    .groupby("domain_key")["score"]
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)
x_positions = np.arange(len(domain_order))
bar_width = 0.36

fig, ax = plt.subplots(figsize=(4.8, 2.8))
for metric_idx, metric in enumerate(PAPER_METRICS):
    stats = (
        domain_long.loc[domain_long["metric"].eq(metric)]
        .groupby("domain_key")["score"]
        .agg(["mean", "std", "count"])
        .reindex(domain_order)
    )
    sem = stats["std"].fillna(0) / np.sqrt(stats["count"].clip(lower=1))
    offsets = x_positions + (metric_idx - (len(PAPER_METRICS) - 1) / 2) * bar_width
    ax.bar(
        offsets,
        stats["mean"],
        yerr=sem,
        width=bar_width * 0.9,
        color=METRIC_PALETTE[metric],
        label=metric,
        error_kw={"linewidth": 0.8, "capsize": 2},
    )
ax.set_ylim(0, 1.02)
ax.set_xticks(x_positions)
ax.set_xticklabels([domain.title() for domain in domain_order])
ax.set_xlabel("")
ax.set_ylabel("Mean score")
ax.legend(title="", frameon=False)
ax.grid(axis="y", alpha=0.35)
despine(ax)
fig.tight_layout()
save_publication_figure(fig, "analysis_domain_robustness")
plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.png
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/2485995464.py:46: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Analysis Figure: Multilingual Tool-Mix Rows

`Tool_Mix_*` rows are analyzed separately from natural-language rows.


In [61]:
tool_mix_df = plot_df.loc[plot_df["language_group"].eq("tool_mix")].copy()

if tool_mix_df.empty:
    print("No Tool_Mix rows with primary metrics were found.")
else:
    tool_mix_long = tool_mix_df.melt(
        id_vars=["tool_mix_n", "language_key", "model_key", "model_label", "domain_key"],
        value_vars=PAPER_METRICS,
        var_name="metric",
        value_name="score",
    ).dropna(subset=["tool_mix_n", "score"])
    tool_mix_long["tool_mix_label"] = tool_mix_long["tool_mix_n"].astype(int).map(lambda n: f"Mix {n}")

    tool_mix_summary = (
        tool_mix_long.groupby(["tool_mix_n", "model_key", "metric"])["score"]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    tool_mix_summary["sem"] = tool_mix_summary["std"].fillna(0) / np.sqrt(
        tool_mix_summary["count"].clip(lower=1)
    )

    models = [model for model in PLOT_MODEL_KEYS if model in set(tool_mix_summary["model_key"])]
    metric_styles = {
        "pass@1": {"marker": "o", "linestyle": "-"},
        "pass@3": {"marker": "X", "linestyle": "--"},
    }

    fig, ax = plt.subplots(figsize=(5.6, 3.1))
    for model in models:
        for metric in PAPER_METRICS:
            subset = tool_mix_summary.loc[
                tool_mix_summary["model_key"].eq(model)
                & tool_mix_summary["metric"].eq(metric)
            ].sort_values("tool_mix_n")
            if subset.empty:
                continue
            ax.errorbar(
                subset["tool_mix_n"],
                subset["mean"],
                yerr=subset["sem"],
                color=MODEL_PALETTE.get(model, OKABE_ITO["blue"]),
                linewidth=1.5,
                capsize=2.5,
                label=f"{MODEL_LABELS.get(model, model)} {metric}",
                **metric_styles.get(metric, {"marker": "o", "linestyle": "-"}),
            )
    ticks = sorted(tool_mix_long["tool_mix_n"].dropna().unique())
    ax.set_xticks(ticks)
    ax.set_xticklabels([f"Mix {int(tick)}" for tick in ticks])
    ax.set_ylim(0, 1.02)
    ax.set_xlabel("Tool mix condition")
    ax.set_ylabel("Mean score")
    ax.legend(title="", frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(axis="y", alpha=0.35)
    despine(ax)
    fig.tight_layout()
    save_publication_figure(fig, "analysis_multilingual_tool_mix")
    plt.show()


Saved:
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_multilingual_tool_mix.pdf
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_multilingual_tool_mix.png
  /Users/saksornr/Code/tau3_result_plot/figures/analysis_multilingual_tool_mix.svg


/var/folders/26/1bbvcxzx3z1f7hghb9kx4hzc0000gn/T/ipykernel_33308/3426591088.py:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export Manifest

Run this cell after plotting to list every generated artifact.


In [62]:
exported_files = sorted(
    path
    for path in FIG_DIR.iterdir()
    if path.suffix.lower() in {".pdf", ".png", ".svg"}
)
print(f"Exported {len(exported_files)} files to {FIG_DIR}")
for path in exported_files:
    print(path)


Exported 27 files to /Users/saksornr/Code/tau3_result_plot/figures
/Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.pdf
/Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.png
/Users/saksornr/Code/tau3_result_plot/figures/analysis_domain_robustness.svg
/Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.pdf
/Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.png
/Users/saksornr/Code/tau3_result_plot/figures/analysis_gap_to_english.svg
/Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.pdf
/Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.png
/Users/saksornr/Code/tau3_result_plot/figures/analysis_heatmap_language_scenario.svg
/Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.pdf
/Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.png
/Users/saksornr/Code/tau3_result_plot/figures/analysis_model_ranking.svg
/Users